# BTIS3043 Artificial Intelligence - Final Assessment
## Student Name: Chew Rong Bin
## Student ID: B240074A

### Project Overview
This notebook implements an intelligent system to query, filter, and rank academic reference eBooks based on two scenarios:
1. **Scenario 1**: Artificial Intelligence, Programming, and Mathematical Foundations.
2. **Scenario 2**: Cybersecurity and Secure Computing.

The system utilizes:
- **Predicate Reasoning**: To identify records satisfying crisp conditions (e.g., specific subject keywords).
- **Fuzzy Reasoning**: To evaluate gradual preferences (e.g., recency, relevance, and affordability).

In [10]:
import pandas as pd
import numpy as np
import os
import re

# Automatically locate data folder
data_dir = 'data' if os.path.exists('data') else '.'

df_a = pd.read_excel(os.path.join(data_dir, 'BTIS3043_Dataset_A_Existing_eBook_Collection.xlsx'), sheet_name='Booklist')
df_b = pd.read_excel(os.path.join(data_dir, 'BTIS3043_Dataset_B_Academic_eBook_Catalogue.xlsx'), sheet_name='Booklist')
df_c = pd.read_excel(os.path.join(data_dir, 'BTIS3043_Dataset_C_eBook_Acquisition_Catalogue.xlsx'), sheet_name='Booklist')

print("Successfully loaded all datasets:")
print(f"Dataset A: {df_a.shape}")
print(f"Dataset B: {df_b.shape}")
print(f"Dataset C: {df_c.shape}")

Successfully loaded all datasets:
Dataset A: (9, 11)
Dataset B: (1743, 13)
Dataset C: (807, 30)


## 2. Fixed Scenario 1: Artificial Intelligence, Programming and Mathematical Foundations
- **Objective**: Query all three datasets for AI, programming, and mathematical foundations.
- **Method**: Predicate filtering followed by fuzzy reasoning (relevance, recency, affordability).

In [11]:
# Define keywords for Scenario 1
ai_keywords = ['Artificial Intelligence', 'Intelligent Systems', 'Machine Learning', 'Expert Systems', 'Robotics']
support_keywords = ['Python', 'Java', 'Algorithms', 'Data Structures', 'Statistics', 'Probability', 'Linear Algebra', 'Calculus']
s1_keywords = ai_keywords + support_keywords

# Predicate Filtering Function
def filter_dataset(df, text_cols, keywords):
    pattern = '|'.join([r'\b' + re.escape(kw) + r'\b' for kw in keywords])
    mask = False
    for col in text_cols:
        if col in df.columns:
            mask = mask | df[col].str.contains(pattern, case=False, na=False)
    return df[mask].copy()

# Execute predicate queries for Scenario 1
res_a_s1 = filter_dataset(df_a, ['Title'], s1_keywords)
res_b_s1 = filter_dataset(df_b, ['Title', 'Discipline (Level 1)', 'Discipline (Level 2)'], s1_keywords)
res_c_s1 = filter_dataset(df_c, ['Title', 'Category', 'Discipline'], s1_keywords)

print(f"Scenario 1 Matches -> Dataset A: {len(res_a_s1)}, Dataset B: {len(res_b_s1)}, Dataset C: {len(res_c_s1)}")

Scenario 1 Matches -> Dataset A: 0, Dataset B: 173, Dataset C: 72


## 2. Fixed Scenario 1: Artificial Intelligence, Programming and Mathematical Foundations
- **Objective**: Query all three datasets for books directly related to Artificial Intelligence, as well as essential programming and mathematical foundations supporting intelligent systems teaching.
- **Method**: Apply Boolean predicate matching across title, category, and discipline attributes to identify candidate references.

In [12]:
def compute_fuzzy_score(title, year, price, direct_kws, support_kws):
    title_str = str(title).lower()
    
    # 1. Relevance Score
    is_direct = any(dk.lower() in title_str for dk in direct_kws)
    support_count = sum(1 for sk in support_kws if sk.lower() in title_str)
    
    if is_direct:
        rel_score = 1.0
    elif support_count > 0:
        rel_score = min(0.5 + (0.2 * support_count), 0.85)
    else:
        rel_score =.2
        
    # 2. Recency Score (Base year 2026)
    try:
        y = int(year)
        age = 2026 - y
        rec_score = 1.0 if age <= 2 else (0.8 if age <= 5 else (0.5 if age <= 10 else 0.2))
    except:
        rec_score = 0.5
        
    # 3. Affordability Score (if price available)
    try:
        p = float(price)
        aff_score = 1.0 if p <= 50 else max(0.1, 1.0 - (p / 300.0))
    except:
        aff_score = 0.5 # Neutral if no price
        
    # Combined Fuzzy Score (Weights: 50% Relevance, 30% Recency, 20% Affordability)
    final_score = (0.5 * rel_score) + (0.3 * rec_score) + (0.2 * aff_score)
    return final_score

# Apply to Dataset A
if len(res_a_s1) > 0:
    res_a_s1['Fuzzy_Score'] = res_a_s1.apply(lambda r: compute_fuzzy_score(r['Title'], r.get('Copyright Year', 2020), r.get('Unit Net Price', 100), ai_keywords, support_keywords), axis=1)
    print("Dataset A Top Ranked (Scenario 1):")
    display(res_a_s1.sort_values(by='Fuzzy_Score', ascending=False)[['Title', 'Copyright Year', 'Fuzzy_Score']].head(5))

## 3. Fuzzy Reasoning and Ranking (Scenario 1)
- **Objective**: Evaluate the gradual suitability of filtered records based on topic depth (direct AI vs. support), publication recency, and price affordability.
- **Method**: Implement custom fuzzy membership functions and compute weighted aggregate scores to rank the top reference books.

In [14]:
# Define comprehensive security keywords
security_keywords = [
    'Cybersecurity', 'Security', 'Cryptography', 'Forensics', 
    'Network Security', 'Information Assurance', 'Secure'
]

# Execute predicate query across all three datasets using the flexible filter_dataset function
res_a_s2 = filter_dataset(df_a, ['Title'], security_keywords)
res_b_s2 = filter_dataset(df_b, ['Title', 'Discipline (Level 1)', 'Discipline (Level 2)'], security_keywords)
res_c_s2 = filter_dataset(df_c, ['Title', 'Category', 'Discipline'], security_keywords)

print(f"Scenario 2 Matches -> Dataset A: {len(res_a_s2)}, Dataset B: {len(res_b_s2)}, Dataset C: {len(res_c_s2)}")

# Preview Dataset B matches for Cybersecurity
if len(res_b_s2) > 0:
    display(res_b_s2[['Title', 'Author', 'Pub Date']].head(5))

Scenario 2 Matches -> Dataset A: 1, Dataset B: 9, Dataset C: 7


,Title,Author,Pub Date
200,"Boyle: Corporate Computer Security, Global Edi...",Boyle,2015-01-23 00:00:00
421,"Computer Security: Principles and Practice, Gl...",Stallings,2024-10-15 00:00:00
422,"Computer Security: Principles and Practice, Gl...",Stallings,2018-06-21 00:00:00
430,Cryptography and Network Security: Principles ...,Stallings,2022-05-24 00:00:00
848,Introduction to Computer Security,Goodrich,2013-08-29 00:00:00


## 4. Fixed Scenario 2: Cybersecurity and Secure Computing
- **Objective**: Review current and potential reference eBooks relating to cybersecurity, network security, cryptography, and digital forensics.
- **Method**: Execute targeted multi-field predicate searches across title, category, and discipline attributes for all three catalogues.

In [15]:
security_keywords = ['Cybersecurity', 'Security', 'Cryptography', 'Forensics', 'Network', 'Information Assurance']

res_a_s2 = filter_dataset(df_a, ['Title'], security_keywords)
res_b_s2 = filter_dataset(df_b, ['Title', 'Discipline (Level 1)', 'Discipline (Level 2)'], security_keywords)
res_c_s2 = filter_dataset(df_c, ['Title', 'Category', 'Discipline'], security_keywords)

print(f"Scenario 2 Matches -> Dataset A: {len(res_a_s2)}, Dataset B: {len(res_b_s2)}, Dataset C: {len(res_c_s2)}")

Scenario 2 Matches -> Dataset A: 1, Dataset B: 9, Dataset C: 9


## 5. Comparison, Analysis and Conclusion
- **Objective**: Contrast the strict binary outputs of predicate-only queries with the graded, preference-ordered results of fuzzy-enhanced queries.
- **Method**: Analyze how dataset size, metadata structure (such as granular disciplines in Dataset B/C vs. flat lists in Dataset A), and price fields influence the overall system outputs.